In [1]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from PIL import Image, ImageEnhance

from sklearn.mixture import GaussianMixture  
from sklearn.metrics import silhouette_score
from sklearn import metrics
from scipy.spatial.distance import cdist

In [ ]:
from __future__ import annotations

import csv
import gc
from pathlib import Path
from typing import Iterable, Tuple

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from skimage.color import rgb2lab
from skimage.filters import gaussian
from skimage.segmentation import mark_boundaries, slic

from rgb_xyz_slic_no_uv import (run_rgb_xyz_slic, mesh_surface_spatial_step)

### SLIC RGB + compactness* XYZ


In [ ]:
# =============================================================================
# USER SETTINGS
# =============================================================================
TEXTURE_PATH = "Texture path.png"
TEXTURE_FILE = Path(TEXTURE_PATH)

Part_identifier = "Part A"  # "Part A" or "Part B"

OUTPUT_DIR = TEXTURE_FILE.parent / "slic_RGBXYZ_Part A_c0_3 DELETE"

########## CHANGE L_STAR_SCALE AND N_KMEANS_CLUSTERS FOR PART A OR PART B RESULTS ##########
# Article parameters for the superpixel stage.
N_SUPERPIXELS = 10_000
COMPACTNESS = 0.3 # 10.0
MAX_SLIC_ITERATIONS = 10
RGB_SMOOTHING_SIGMA = 0 #RGB_SMOOTHING_SIGMA = 1.0

# Weight of the normalized XYZ distance relative to normalized RGB distance.
# 0.0 reproduces texture-only SLIC; 1.0 gives RGB and XYZ equal channel weight.
# XYZ_WEIGHT = 1.0

# OBJ V coordinates normally start at the bottom, whereas image rows start at
# the top. Keep True unless the texture appears vertically inverted.
FLIP_V = True

if Part_identifier == "Part A":
    OBJ_FILE = Path(r"MODEL_PATH\Mushroom_PartA.obj")

    # The article used 0.6 for Part B and 0.7 for Part A.
    L_STAR_SCALE = 0.7 # L_STAR_SCALE = 0.7

    # Kept at 5 to preserve your earlier request. For the article's Part B result,
    # change this to 3. For Part A, use 4.
    N_KMEANS_CLUSTERS = 4

elif Part_identifier == "Part B":
    OBJ_FILE = Path(r"MODEL_PATH\Mushroom_PartB.obj")

    L_STAR_SCALE = 0.6 
    N_KMEANS_CLUSTERS = 3

KMEANS_N_INIT = 20
RANDOM_STATE = 0

RUN_K_DIAGNOSTICS = False
K_DIAGNOSTIC_VALUES = tuple(range(2, 11))
SILHOUETTE_SAMPLE_SIZE = 5_000

SELECT_REPRESENTATIVE_POINTS = False
POINTS_PER_CLUSTER = 8

SHOW_FIGURES = True


# =============================================================================
# OBJ LOADING
# =============================================================================
def _resolve_obj_index(index_text: str, current_length: int) -> int:
    """Convert an OBJ one-based or negative index to a zero-based index."""
    index = int(index_text)
    resolved = index - 1 if index > 0 else current_length + index
    if resolved < 0 or resolved >= current_length:
        raise IndexError(
            f"OBJ index {index} resolves to {resolved}, outside 0..{current_length - 1}."
        )
    return resolved


def load_obj_vertices_uv_triangles(
    obj_path: Path,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Load vertices, UVs, and triangulated faces from an OBJ file.

    Returns
    -------
    vertices : (Nv, 3) float32
    texcoords : (Nt, 2) float32
    triangle_vertex_indices : (Nf, 3) int32
    triangle_uv_indices : (Nf, 3) int32
    """
    if not obj_path.is_file():
        raise FileNotFoundError(f"OBJ file not found: {obj_path}")

    vertices: list[list[float]] = []
    texcoords: list[list[float]] = []
    triangle_vertex_indices: list[list[int]] = []
    triangle_uv_indices: list[list[int]] = []

    with obj_path.open("r", encoding="utf-8", errors="ignore") as file:
        for line_number, raw_line in enumerate(file, start=1):
            line = raw_line.strip()
            if not line or line.startswith("#"):
                continue

            fields = line.split()
            record_type = fields[0]

            try:
                if record_type == "v" and len(fields) >= 4:
                    vertices.append(
                        [float(fields[1]), float(fields[2]), float(fields[3])]
                    )

                elif record_type == "vt" and len(fields) >= 3:
                    texcoords.append([float(fields[1]), float(fields[2])])

                elif record_type == "f" and len(fields) >= 4:
                    face_vertex_indices: list[int] = []
                    face_uv_indices: list[int] = []

                    for reference in fields[1:]:
                        parts = reference.split("/")
                        if not parts[0] or len(parts) < 2 or not parts[1]:
                            face_vertex_indices = []
                            face_uv_indices = []
                            break

                        face_vertex_indices.append(
                            _resolve_obj_index(parts[0], len(vertices))
                        )
                        face_uv_indices.append(
                            _resolve_obj_index(parts[1], len(texcoords))
                        )

                    # Fan triangulation supports triangles, quads, and n-gons.
                    for corner in range(1, len(face_vertex_indices) - 1):
                        triangle_vertex_indices.append(
                            [
                                face_vertex_indices[0],
                                face_vertex_indices[corner],
                                face_vertex_indices[corner + 1],
                            ]
                        )
                        triangle_uv_indices.append(
                            [
                                face_uv_indices[0],
                                face_uv_indices[corner],
                                face_uv_indices[corner + 1],
                            ]
                        )
            except (ValueError, IndexError) as exc:
                raise ValueError(
                    f"Failed to parse OBJ line {line_number}: {raw_line.rstrip()}"
                ) from exc

    if not vertices:
        raise ValueError(f"No vertices were found in {obj_path}")
    if not texcoords:
        raise ValueError(f"No texture coordinates were found in {obj_path}")
    if not triangle_vertex_indices:
        raise ValueError(
            "No faces with both vertex and UV indices were found in the OBJ."
        )

    return (
        np.asarray(vertices, dtype=np.float32),
        np.asarray(texcoords, dtype=np.float32),
        np.asarray(triangle_vertex_indices, dtype=np.int32),
        np.asarray(triangle_uv_indices, dtype=np.int32),
    )


# =============================================================================
# UV RASTERIZATION: TEXTURE PIXEL -> XYZ
# =============================================================================
def rasterize_xyz_map(
    vertices: np.ndarray,
    texcoords: np.ndarray,
    triangle_vertex_indices: np.ndarray,
    triangle_uv_indices: np.ndarray,
    image_height: int,
    image_width: int,
    flip_v: bool,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Rasterize UV triangles and barycentrically interpolate mesh XYZ."""
    xyz_sum = np.zeros((image_height, image_width, 3), dtype=np.float32)
    hit_count = np.zeros((image_height, image_width), dtype=np.uint32)

    uv_min = texcoords.min(axis=0)
    uv_max = texcoords.max(axis=0)
    if np.any(uv_min < -1.0e-5) or np.any(uv_max > 1.0 + 1.0e-5):
        print(
            "Warning: some UV coordinates lie outside [0, 1]. "
            "Only their intersection with the texture is rasterized."
        )

    number_of_triangles = len(triangle_vertex_indices)

    for triangle_number, (vertex_ids, uv_ids) in enumerate(
        zip(triangle_vertex_indices, triangle_uv_indices), start=1
    ):
        uv = texcoords[uv_ids].astype(np.float64, copy=False)
        xyz = vertices[vertex_ids].astype(np.float64, copy=False)

        x = uv[:, 0] * (image_width - 1)
        if flip_v:
            y = (1.0 - uv[:, 1]) * (image_height - 1)
        else:
            y = uv[:, 1] * (image_height - 1)

        minimum_x = max(0, int(np.floor(x.min())))
        maximum_x = min(image_width - 1, int(np.ceil(x.max())))
        minimum_y = max(0, int(np.floor(y.min())))
        maximum_y = min(image_height - 1, int(np.ceil(y.max())))

        if maximum_x < minimum_x or maximum_y < minimum_y:
            continue

        denominator = (
            (y[1] - y[2]) * (x[0] - x[2])
            + (x[2] - x[1]) * (y[0] - y[2])
        )
        if abs(denominator) < 1.0e-14:
            continue

        grid_x, grid_y = np.meshgrid(
            np.arange(minimum_x, maximum_x + 1, dtype=np.float64),
            np.arange(minimum_y, maximum_y + 1, dtype=np.float64),
        )

        weight_0 = (
            (y[1] - y[2]) * (grid_x - x[2])
            + (x[2] - x[1]) * (grid_y - y[2])
        ) / denominator
        weight_1 = (
            (y[2] - y[0]) * (grid_x - x[2])
            + (x[0] - x[2]) * (grid_y - y[2])
        ) / denominator
        weight_2 = 1.0 - weight_0 - weight_1

        tolerance = 1.0e-7
        inside = (
            (weight_0 >= -tolerance)
            & (weight_1 >= -tolerance)
            & (weight_2 >= -tolerance)
        )
        if not inside.any():
            continue

        local_y, local_x = np.nonzero(inside)
        pixel_y = local_y + minimum_y
        pixel_x = local_x + minimum_x

        interpolated_xyz = (
            weight_0[inside, None] * xyz[0]
            + weight_1[inside, None] * xyz[1]
            + weight_2[inside, None] * xyz[2]
        ).astype(np.float32)

        xyz_sum[pixel_y, pixel_x] += interpolated_xyz
        hit_count[pixel_y, pixel_x] += 1

        if triangle_number % 10_000 == 0 or triangle_number == number_of_triangles:
            print(
                f"Rasterized {triangle_number:,}/{number_of_triangles:,} UV triangles",
                flush=True,
            )

    uv_mask = hit_count > 0
    if not uv_mask.any():
        raise ValueError(
            "No UV triangle covered the texture. Check the OBJ UVs and FLIP_V."
        )

    xyz_map = np.zeros_like(xyz_sum)
    xyz_map[uv_mask] = (
        xyz_sum[uv_mask] / hit_count[uv_mask, None].astype(np.float32)
    )

    return xyz_map, uv_mask, hit_count

def rasterize_xyz_map_updated(
    vertices: np.ndarray,
    texcoords: np.ndarray,
    triangle_vertex_indices: np.ndarray,
    triangle_uv_indices: np.ndarray,
    image_height: int,
    image_width: int,
    flip_v: bool,
    distance_tolerance: float | None = None,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Rasterize UV triangles and interpolate mesh XYZ.

    Pixels receiving geometrically distant XYZ values are marked as ambiguous.

    Returns
    -------
    xyz_map
        Array (H, W, 3) containing one XYZ coordinate per valid pixel.

    uv_mask
        Pixels with a valid and unambiguous XYZ coordinate.

    hit_count
        Number of triangles covering each texture pixel.

    ambiguous_mask
        Pixels associated with geometrically distant XYZ positions.
    """

    xyz_sum = np.zeros(
        (image_height, image_width, 3),
        dtype=np.float32,
    )

    xyz_reference = np.zeros(
        (image_height, image_width, 3),
        dtype=np.float32,
    )

    hit_count = np.zeros(
        (image_height, image_width),
        dtype=np.uint32,
    )

    ambiguous_mask = np.zeros(
        (image_height, image_width),
        dtype=bool,
    )

    # Characteristic size of the model.
    bounding_box_size = np.linalg.norm(
        vertices.max(axis=0) - vertices.min(axis=0)
    )

    if distance_tolerance is None:
        distance_tolerance = max(
            1.0e-6,
            1.0e-5 * bounding_box_size,
        )

    print(
        f"XYZ overlap tolerance: {distance_tolerance:.6g}"
    )

    uv_min = texcoords.min(axis=0)
    uv_max = texcoords.max(axis=0)

    if np.any(uv_min < -1.0e-5) or np.any(uv_max > 1.0 + 1.0e-5):
        print(
            "Warning: some UV coordinates lie outside [0, 1]. "
            "Only their intersection with the texture is rasterized."
        )

    number_of_triangles = len(triangle_vertex_indices)

    for triangle_number, (vertex_ids, uv_ids) in enumerate(
        zip(triangle_vertex_indices, triangle_uv_indices),
        start=1,
    ):
        uv = texcoords[uv_ids].astype(
            np.float64,
            copy=False,
        )

        xyz = vertices[vertex_ids].astype(
            np.float64,
            copy=False,
        )

        x = uv[:, 0] * (image_width - 1)

        if flip_v:
            y = (1.0 - uv[:, 1]) * (image_height - 1)
        else:
            y = uv[:, 1] * (image_height - 1)

        minimum_x = max(
            0,
            int(np.floor(x.min())),
        )

        maximum_x = min(
            image_width - 1,
            int(np.ceil(x.max())),
        )

        minimum_y = max(
            0,
            int(np.floor(y.min())),
        )

        maximum_y = min(
            image_height - 1,
            int(np.ceil(y.max())),
        )

        if maximum_x < minimum_x or maximum_y < minimum_y:
            continue

        denominator = (
            (y[1] - y[2]) * (x[0] - x[2])
            + (x[2] - x[1]) * (y[0] - y[2])
        )

        if abs(denominator) < 1.0e-14:
            continue

        grid_x, grid_y = np.meshgrid(
            np.arange(
                minimum_x,
                maximum_x + 1,
                dtype=np.float64,
            ),
            np.arange(
                minimum_y,
                maximum_y + 1,
                dtype=np.float64,
            ),
        )

        weight_0 = (
            (y[1] - y[2]) * (grid_x - x[2])
            + (x[2] - x[1]) * (grid_y - y[2])
        ) / denominator

        weight_1 = (
            (y[2] - y[0]) * (grid_x - x[2])
            + (x[0] - x[2]) * (grid_y - y[2])
        ) / denominator

        weight_2 = 1.0 - weight_0 - weight_1

        tolerance = 1.0e-7

        inside = (
            (weight_0 >= -tolerance)
            & (weight_1 >= -tolerance)
            & (weight_2 >= -tolerance)
        )

        if not inside.any():
            continue

        local_y, local_x = np.nonzero(inside)

        pixel_y = local_y + minimum_y
        pixel_x = local_x + minimum_x

        interpolated_xyz = (
            weight_0[inside, None] * xyz[0]
            + weight_1[inside, None] * xyz[1]
            + weight_2[inside, None] * xyz[2]
        ).astype(np.float32)

        previous_hit_count = hit_count[pixel_y, pixel_x]

        first_hit = previous_hit_count == 0
        repeated_hit = ~first_hit

        # Store the first XYZ found for each pixel.
        if first_hit.any():
            first_y = pixel_y[first_hit]
            first_x = pixel_x[first_hit]

            xyz_reference[first_y, first_x] = interpolated_xyz[first_hit]

        # Compare later hits with the first XYZ.
        if repeated_hit.any():
            repeated_y = pixel_y[repeated_hit]
            repeated_x = pixel_x[repeated_hit]

            previous_xyz = xyz_reference[
                repeated_y,
                repeated_x,
            ]

            new_xyz = interpolated_xyz[repeated_hit]

            xyz_distance = np.linalg.norm(
                new_xyz - previous_xyz,
                axis=1,
            )

            distant_hit = xyz_distance > distance_tolerance

            if distant_hit.any():
                ambiguous_y = repeated_y[distant_hit]
                ambiguous_x = repeated_x[distant_hit]

                ambiguous_mask[
                    ambiguous_y,
                    ambiguous_x,
                ] = True

        xyz_sum[pixel_y, pixel_x] += interpolated_xyz
        hit_count[pixel_y, pixel_x] += 1

        if (
            triangle_number % 10_000 == 0
            or triangle_number == number_of_triangles
        ):
            print(
                f"Rasterized "
                f"{triangle_number:,}/{number_of_triangles:,} "
                f"UV triangles",
                flush=True,
            )

    covered_mask = hit_count > 0

    if not covered_mask.any():
        raise ValueError(
            "No UV triangle covered the texture. "
            "Check the OBJ UVs and flip_v."
        )

    # Only unambiguous pixels receive a single XYZ coordinate.
    uv_mask = covered_mask & ~ambiguous_mask

    xyz_map = np.zeros_like(xyz_sum)

    xyz_map[uv_mask] = (
        xyz_sum[uv_mask]
        / hit_count[uv_mask, None].astype(np.float32)
    )

    print(f"Covered pixels: {covered_mask.sum():,}")
    print(f"Valid XYZ pixels: {uv_mask.sum():,}")
    print(f"Ambiguous UV pixels: {ambiguous_mask.sum():,}")
    print(f"Maximum hit count: {hit_count.max()}")

    return (xyz_map, uv_mask, hit_count, ambiguous_mask, )

# =============================================================================
# GENERALIZED SLIC: RGB + XYZ + BUILT-IN UV SPATIAL TERM
# =============================================================================

## Imported from the rgb_xyz_slic_no_uv.py


# =============================================================================
# ARTICLE SUPERPIXEL SMOOTHING: MEAN RGB
# =============================================================================
def compute_superpixel_means(
    rgb: np.ndarray,
    xyz_map: np.ndarray,
    labels: np.ndarray,
    valid_mask: np.ndarray,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Compute mean RGB, mean XYZ, mean pixel position, and area per superpixel."""
    valid_labels = labels[valid_mask]
    if valid_labels.size == 0 or valid_labels.min() < 0:
        raise ValueError("No valid superpixel labels are available.")

    number_of_superpixels = int(valid_labels.max()) + 1
    counts = np.bincount(valid_labels, minlength=number_of_superpixels).astype(
        np.int64
    )
    if np.any(counts == 0):
        raise RuntimeError("Contiguous relabeling produced an empty superpixel.")

    mean_rgb = np.zeros((number_of_superpixels, 3), dtype=np.float32)
    mean_xyz = np.zeros((number_of_superpixels, 3), dtype=np.float32)

    for channel in range(3):
        mean_rgb[:, channel] = np.bincount(
            valid_labels,
            weights=rgb[..., channel][valid_mask],
            minlength=number_of_superpixels,
        ) / counts
        mean_xyz[:, channel] = np.bincount(
            valid_labels,
            weights=xyz_map[..., channel][valid_mask],
            minlength=number_of_superpixels,
        ) / counts

    rows, columns = np.nonzero(valid_mask)
    mean_row = np.bincount(
        valid_labels, weights=rows, minlength=number_of_superpixels
    ) / counts
    mean_column = np.bincount(
        valid_labels, weights=columns, minlength=number_of_superpixels
    ) / counts
    mean_pixel_rc = np.column_stack([mean_row, mean_column]).astype(np.float32)

    mean_rgb_texture = np.zeros_like(rgb, dtype=np.float32)
    mean_rgb_texture[valid_mask] = mean_rgb[valid_labels]

    return mean_rgb_texture, mean_rgb, mean_xyz, mean_pixel_rc, counts


def superpixel_rgb_to_adjusted_lab(
    mean_rgb: np.ndarray,
    l_star_scale: float,
) -> Tuple[np.ndarray, np.ndarray]:
    """Convert mean RGB colors to Lab and apply the article's L* scaling."""
    if l_star_scale <= 0:
        raise ValueError("L_STAR_SCALE must be positive.")

    lab_original = rgb2lab(mean_rgb.reshape(1, -1, 3)).reshape(-1, 3).astype(
        np.float32
    )
    lab_for_kmeans = lab_original.copy()
    lab_for_kmeans[:, 0] *= float(l_star_scale)
    return lab_original, lab_for_kmeans


# =============================================================================
# K-MEANS AND K DIAGNOSTICS
# =============================================================================
def run_k_diagnostics(
    lab_features: np.ndarray,
    k_values: Iterable[int],
    random_state: int,
    output_dir: Path,
) -> list[dict[str, float]]:
    """Compute article-style elbow inertia and silhouette scores."""
    results: list[dict[str, float]] = []
    number_of_samples = len(lab_features)

    for k in k_values:
        if k < 2 or k >= number_of_samples:
            continue

        model = KMeans(
            n_clusters=k,
            n_init=KMEANS_N_INIT,
            random_state=random_state,
        )
        labels = model.fit_predict(lab_features)

        silhouette = silhouette_score(
            lab_features,
            labels,
            sample_size=min(SILHOUETTE_SAMPLE_SIZE, number_of_samples),
            random_state=random_state,
        )
        results.append(
            {"k": float(k), "inertia": float(model.inertia_), "silhouette": float(silhouette)}
        )
        print(
            f"k={k}: inertia={model.inertia_:.3f}, silhouette={silhouette:.4f}"
        )

    if not results:
        print("K diagnostics skipped: no valid k values.")
        return results

    csv_path = output_dir / "k_diagnostics.csv"
    with csv_path.open("w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=["k", "inertia", "silhouette"])
        writer.writeheader()
        writer.writerows(results)

    k_array = np.asarray([row["k"] for row in results])
    inertia_array = np.asarray([row["inertia"] for row in results])
    silhouette_array = np.asarray([row["silhouette"] for row in results])

    figure, axes = plt.subplots(1, 2, figsize=(11, 4.5))
    axes[0].plot(k_array, inertia_array, marker="o")
    axes[0].set_xlabel("Number of clusters, k")
    axes[0].set_ylabel("K-means inertia")
    axes[0].set_title("Elbow method")
    axes[0].grid(alpha=0.3)

    axes[1].plot(k_array, silhouette_array, marker="o")
    axes[1].set_xlabel("Number of clusters, k")
    axes[1].set_ylabel("Silhouette score")
    axes[1].set_title("Silhouette analysis")
    axes[1].grid(alpha=0.3)

    figure.tight_layout()
    figure.savefig(output_dir / "k_diagnostics.png", dpi=200, bbox_inches="tight")
    plt.close(figure)

    return results


def cluster_superpixel_lab(
    lab_features: np.ndarray,
    mean_rgb: np.ndarray,
    n_clusters: int,
    random_state: int,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Apply unweighted K-means to one adjusted Lab vector per superpixel."""
    if n_clusters < 2:
        raise ValueError("N_KMEANS_CLUSTERS must be at least 2.")
    if len(lab_features) < n_clusters:
        raise ValueError(
            f"Only {len(lab_features)} superpixels exist; cannot create {n_clusters} clusters."
        )

    model = KMeans(
        n_clusters=n_clusters,
        n_init=KMEANS_N_INIT,
        random_state=random_state,
    )
    raw_cluster_for_superpixel = model.fit_predict(lab_features)
    raw_centers = model.cluster_centers_.astype(np.float32)

    # Stable, interpretable IDs: C0 is darkest, followed by increasing L*.
    center_order = np.argsort(raw_centers[:, 0])
    raw_to_ordered = np.empty_like(center_order)
    raw_to_ordered[center_order] = np.arange(n_clusters)
    cluster_for_superpixel = raw_to_ordered[raw_cluster_for_superpixel].astype(
        np.int32
    )
    ordered_centers_lab = raw_centers[center_order]

    # Natural display color for each final cluster, computed from assigned
    # superpixel mean RGB values. K-means itself still uses adjusted Lab.
    cluster_display_rgb = np.zeros((n_clusters, 3), dtype=np.float32)
    for cluster_id in range(n_clusters):
        members = cluster_for_superpixel == cluster_id
        cluster_display_rgb[cluster_id] = mean_rgb[members].mean(axis=0)

    return cluster_for_superpixel, ordered_centers_lab, cluster_display_rgb


def propagate_cluster_labels(
    labels: np.ndarray,
    valid_mask: np.ndarray,
    cluster_for_superpixel: np.ndarray,
    cluster_display_rgb: np.ndarray,
) -> Tuple[np.ndarray, np.ndarray]:
    """Map each superpixel cluster label and representative color to texels."""
    cluster_map = np.full(labels.shape, -1, dtype=np.int32)
    cluster_map[valid_mask] = cluster_for_superpixel[labels[valid_mask]]

    cluster_rgb_texture = np.zeros((*labels.shape, 3), dtype=np.float32)
    cluster_rgb_texture[valid_mask] = cluster_display_rgb[cluster_map[valid_mask]]
    return cluster_map, cluster_rgb_texture


# =============================================================================
# REPRESENTATIVE 3D SAMPLING POINTS
# =============================================================================
def _farthest_point_indices(points: np.ndarray, number_to_select: int) -> np.ndarray:
    """Select spatially distributed point indices using greedy farthest sampling."""
    number_to_select = min(number_to_select, len(points))
    if number_to_select <= 0:
        return np.empty(0, dtype=np.int32)

    center = points.mean(axis=0)
    first = int(np.argmin(np.sum((points - center) ** 2, axis=1)))
    selected = [first]
    minimum_squared_distance = np.sum((points - points[first]) ** 2, axis=1)

    for _ in range(1, number_to_select):
        next_index = int(np.argmax(minimum_squared_distance))
        selected.append(next_index)
        squared_distance = np.sum((points - points[next_index]) ** 2, axis=1)
        minimum_squared_distance = np.minimum(
            minimum_squared_distance, squared_distance
        )

    return np.asarray(selected, dtype=np.int32)


def select_representative_points(
    cluster_for_superpixel: np.ndarray,
    mean_xyz: np.ndarray,
    mean_pixel_rc: np.ndarray,
    image_height: int,
    image_width: int,
    points_per_cluster: int,
) -> list[dict[str, float | int]]:
    """Select spatially distinct superpixel centroids inside every cluster."""
    rows: list[dict[str, float | int]] = []
    number_of_clusters = int(cluster_for_superpixel.max()) + 1

    for cluster_id in range(number_of_clusters):
        superpixel_ids = np.flatnonzero(cluster_for_superpixel == cluster_id)
        selected_local_ids = _farthest_point_indices(
            mean_xyz[superpixel_ids], points_per_cluster
        )

        for point_index, local_id in enumerate(selected_local_ids, start=1):
            superpixel_id = int(superpixel_ids[local_id])
            row, column = mean_pixel_rc[superpixel_id]
            x, y, z = mean_xyz[superpixel_id]
            rows.append(
                {
                    "cluster": cluster_id,
                    "point_in_cluster": point_index,
                    "superpixel": superpixel_id,
                    "pixel_x": float(column),
                    "pixel_y": float(row),
                    "u": float(column / max(image_width - 1, 1)),
                    "v_image": float(row / max(image_height - 1, 1)),
                    "v_obj": float(1.0 - row / max(image_height - 1, 1)),
                    "x": float(x),
                    "y": float(y),
                    "z": float(z),
                }
            )

    return rows


# =============================================================================
# OUTPUT HELPERS
# =============================================================================
def save_rgba_float_image(
    path: Path,
    rgb: np.ndarray,
    alpha: np.ndarray,
    valid_mask: np.ndarray,
) -> None:
    """Save an RGB float image while retaining transparency outside the mesh."""
    output_alpha = np.where(valid_mask, alpha, 0.0)
    rgba = np.dstack([np.clip(rgb, 0.0, 1.0), np.clip(output_alpha, 0.0, 1.0)])
    Image.fromarray(np.round(rgba * 255.0).astype(np.uint8), mode="RGBA").save(path)


def make_categorical_cluster_rgba(
    cluster_map: np.ndarray,
    valid_mask: np.ndarray,
    alpha: np.ndarray,
    number_of_clusters: int,
) -> np.ndarray:
    """Create a categorical RGBA visualization of the cluster map."""
    colormap = plt.get_cmap("tab10", number_of_clusters)
    rgba = np.zeros((*cluster_map.shape, 4), dtype=np.float32)
    rgba[valid_mask] = colormap(cluster_map[valid_mask])
    rgba[..., 3] = np.where(valid_mask, alpha, 0.0)
    return rgba


def save_superpixel_table(
    path: Path,
    mean_rgb: np.ndarray,
    lab_original: np.ndarray,
    lab_adjusted: np.ndarray,
    mean_xyz: np.ndarray,
    mean_pixel_rc: np.ndarray,
    counts: np.ndarray,
    cluster_for_superpixel: np.ndarray,
) -> None:
    """Save all superpixel-level features used by the article pipeline."""
    fieldnames = [
        "superpixel",
        "cluster",
        "pixel_count",
        "pixel_x",
        "pixel_y",
        "mean_r",
        "mean_g",
        "mean_b",
        "lab_l_original",
        "lab_a",
        "lab_b",
        "lab_l_adjusted",
        "mean_x",
        "mean_y",
        "mean_z",
    ]
    with path.open("w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()
        for superpixel_id in range(len(mean_rgb)):
            writer.writerow(
                {
                    "superpixel": superpixel_id,
                    "cluster": int(cluster_for_superpixel[superpixel_id]),
                    "pixel_count": int(counts[superpixel_id]),
                    "pixel_x": float(mean_pixel_rc[superpixel_id, 1]),
                    "pixel_y": float(mean_pixel_rc[superpixel_id, 0]),
                    "mean_r": float(mean_rgb[superpixel_id, 0]),
                    "mean_g": float(mean_rgb[superpixel_id, 1]),
                    "mean_b": float(mean_rgb[superpixel_id, 2]),
                    "lab_l_original": float(lab_original[superpixel_id, 0]),
                    "lab_a": float(lab_original[superpixel_id, 1]),
                    "lab_b": float(lab_original[superpixel_id, 2]),
                    "lab_l_adjusted": float(lab_adjusted[superpixel_id, 0]),
                    "mean_x": float(mean_xyz[superpixel_id, 0]),
                    "mean_y": float(mean_xyz[superpixel_id, 1]),
                    "mean_z": float(mean_xyz[superpixel_id, 2]),
                }
            )


def save_cluster_centers(
    path: Path,
    centers_lab_adjusted: np.ndarray,
    cluster_display_rgb: np.ndarray,
) -> None:
    fieldnames = [
        "cluster",
        "center_l_adjusted",
        "center_a",
        "center_b",
        "display_r",
        "display_g",
        "display_b",
    ]
    with path.open("w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()
        for cluster_id in range(len(centers_lab_adjusted)):
            writer.writerow(
                {
                    "cluster": cluster_id,
                    "center_l_adjusted": float(centers_lab_adjusted[cluster_id, 0]),
                    "center_a": float(centers_lab_adjusted[cluster_id, 1]),
                    "center_b": float(centers_lab_adjusted[cluster_id, 2]),
                    "display_r": float(cluster_display_rgb[cluster_id, 0]),
                    "display_g": float(cluster_display_rgb[cluster_id, 1]),
                    "display_b": float(cluster_display_rgb[cluster_id, 2]),
                }
            )


def create_pipeline_figure(
    rgb: np.ndarray,
    alpha: np.ndarray,
    labels: np.ndarray,
    valid_mask: np.ndarray,
    mean_rgb_texture: np.ndarray,
    categorical_rgba: np.ndarray,
    cluster_rgb_texture: np.ndarray,
    representative_points: list[dict[str, float | int]],
    output_dir: Path,
) -> None:
    """Display and save the article-style processing pipeline."""
    overlay_labels = labels.copy()
    overlay_labels[~valid_mask] = int(labels[valid_mask].max()) + 1
    boundary_overlay = mark_boundaries(
        rgb,
        overlay_labels,
        color=(1.0, 0.0, 0.0),
        mode="thick",
    )
    boundary_overlay[~valid_mask] = 0.0

    original_rgba = np.dstack([rgb, np.where(valid_mask, alpha, 0.0)])
    mean_rgba = np.dstack(
        [mean_rgb_texture, np.where(valid_mask, alpha, 0.0)]
    )
    cluster_rgb_rgba = np.dstack(
        [cluster_rgb_texture, np.where(valid_mask, alpha, 0.0)]
    )

    figure, axes = plt.subplots(2, 3, figsize=(18, 11))
    axes[0, 0].imshow(original_rgba)
    axes[0, 0].set_title("A. Original RGB texture")

    axes[0, 1].imshow(boundary_overlay)
    axes[0, 1].set_title("B. SLIC using RGB + XYZ")

    axes[0, 2].imshow(mean_rgba)
    axes[0, 2].set_title("C. Superpixels filled with mean RGB")

    axes[1, 0].imshow(categorical_rgba)
    axes[1, 0].set_title("D. K-means labels from mean CIELAB")

    axes[1, 1].imshow(cluster_rgb_rgba)
    axes[1, 1].set_title("Cluster-centroid color texture")

    axes[1, 2].imshow(categorical_rgba)
    if representative_points:
        for row in representative_points:
            axes[1, 2].plot(
                float(row["pixel_x"]),
                float(row["pixel_y"]),
                marker="o",
                markersize=5,
                markeredgecolor="black",
                markerfacecolor="white",
                linestyle="None",
            )
    axes[1, 2].set_title("Spatially distributed candidate points")

    for axis in axes.ravel():
        axis.axis("off")

    figure.tight_layout()
    figure.savefig(
        output_dir / "pipeline_overview.png", dpi=200, bbox_inches="tight"
    )
    if SHOW_FIGURES:
        plt.show()
    else:
        plt.close(figure)


# =============================================================================
# MAIN
# =============================================================================
def main() -> None:
    if not TEXTURE_FILE.is_file():
        raise FileNotFoundError(f"Texture file not found: {TEXTURE_FILE}")

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    print("1/8 Loading texture...")
    texture = Image.open(TEXTURE_FILE).convert("RGBA")
    rgba = np.asarray(texture, dtype=np.float32) / 255.0

    rgb = rgba[..., :3]
    alpha = rgba[..., 3]
    image_height, image_width = rgb.shape[:2]

    print("2/8 Loading OBJ vertices, UV coordinates, and faces...")
    vertices, texcoords, triangle_vertex_indices, triangle_uv_indices = load_obj_vertices_uv_triangles(OBJ_FILE)
    
    print(f"  vertices: {len(vertices):,}")
    print(f"  UV coordinates: {len(texcoords):,}")
    print(f"  triangulated faces: {len(triangle_vertex_indices):,}")
    print(f"  texture: {image_width:,} x {image_height:,}")

    print("3/8 Rasterizing texture texels to mesh XYZ...")
    xyz_map, uv_mask, hit_count = rasterize_xyz_map(
        vertices=vertices,
        texcoords=texcoords,
        triangle_vertex_indices=triangle_vertex_indices,
        triangle_uv_indices=triangle_uv_indices,
        image_height=image_height,
        image_width=image_width,
        flip_v=FLIP_V,
    )

    # xyz_map, uv_mask, hit_count, ambiguous_mask = rasterize_xyz_map_updated(
    #     vertices=vertices,
    #     texcoords=texcoords,
    #     triangle_vertex_indices=triangle_vertex_indices,
    #     triangle_uv_indices=triangle_uv_indices,
    #     image_height=image_height,
    #     image_width=image_width,
    #     flip_v=FLIP_V,
    # )
    

    alpha_mask = alpha > 0.0
    valid_mask = uv_mask & alpha_mask
    if not valid_mask.any():
        raise ValueError(
            "The intersection of the UV coverage and non-transparent texture is empty."
        )
    print(f"  valid texels: {np.count_nonzero(valid_mask):,}")
    overlapping_texels = int(np.count_nonzero(hit_count > 1))
    if overlapping_texels:
        print(
            f"  warning: {overlapping_texels:,} texels are covered by multiple UV triangles; "
            "their XYZ values were averaged."
        )

    print("4/8 Running generalized SLIC on RGB + XYZ...")
    # For a 2D mesh surface embedded in 3D:
    #
    #     S = sqrt(mesh surface area / number of superpixels)
    #
    # This is the surface equivalent of the normal SLIC interval
    # S = sqrt(number of pixels / number of superpixels).
    spatial_step = mesh_surface_spatial_step(
        vertices=vertices,
        triangle_vertex_indices=triangle_vertex_indices,
        n_segments=N_SUPERPIXELS,
    )

    labels = run_rgb_xyz_slic(
        rgb=rgb,
        xyz=xyz_map,
        valid_mask=valid_mask,
        n_segments=N_SUPERPIXELS,
        compactness=COMPACTNESS,
        max_num_iter=MAX_SLIC_ITERATIONS,
        spatial_step=spatial_step,
        convergence_tol=1.0e-4,
        chunk_size=250_000,
        random_state=RANDOM_STATE,
        verbose=True,
    )


    gc.collect()
    number_of_superpixels = int(labels[valid_mask].max()) + 1
    print(f"  created superpixels: {number_of_superpixels:,}")

    print("5/8 Filling each superpixel with its mean RGB...")
    (
        mean_rgb_texture,
        mean_rgb,
        mean_xyz,
        mean_pixel_rc,
        pixel_counts,
    ) = compute_superpixel_means(
        rgb=rgb,
        xyz_map=xyz_map,
        labels=labels,
        valid_mask=valid_mask,
    )

    print("6/8 Converting mean RGB to CIELAB and running K-means...")
    lab_original, lab_for_kmeans = superpixel_rgb_to_adjusted_lab(
        mean_rgb=mean_rgb,
        l_star_scale=L_STAR_SCALE,
    )

    if RUN_K_DIAGNOSTICS:
        print("  computing elbow and silhouette diagnostics...")
        run_k_diagnostics(
            lab_features=lab_for_kmeans,
            k_values=K_DIAGNOSTIC_VALUES,
            random_state=RANDOM_STATE,
            output_dir=OUTPUT_DIR,
        )

    (
        cluster_for_superpixel,
        centers_lab_adjusted,
        cluster_display_rgb,
    ) = cluster_superpixel_lab(
        lab_features=lab_for_kmeans,
        mean_rgb=mean_rgb,
        n_clusters=N_KMEANS_CLUSTERS,
        random_state=RANDOM_STATE,
    )
    cluster_map, cluster_rgb_texture = propagate_cluster_labels(
        labels=labels,
        valid_mask=valid_mask,
        cluster_for_superpixel=cluster_for_superpixel,
        cluster_display_rgb=cluster_display_rgb,
    )

    print("  adjusted Lab cluster centers:")
    for cluster_id, center in enumerate(centers_lab_adjusted):
        print(
            f"    C{cluster_id}: L={center[0]:.3f}, "
            f"a={center[1]:.3f}, b={center[2]:.3f}"
        )

    print("7/8 Selecting candidate points and saving outputs...")
    representative_points: list[dict[str, float | int]] = []
    if SELECT_REPRESENTATIVE_POINTS:
        representative_points = select_representative_points(
            cluster_for_superpixel=cluster_for_superpixel,
            mean_xyz=mean_xyz,
            mean_pixel_rc=mean_pixel_rc,
            image_height=image_height,
            image_width=image_width,
            points_per_cluster=POINTS_PER_CLUSTER,
        )
        with (OUTPUT_DIR / "representative_points.csv").open(
            "w", newline="", encoding="utf-8"
        ) as file:
            writer = csv.DictWriter(
                file,
                fieldnames=[
                    "cluster", "point_in_cluster", "superpixel", "pixel_x",
                     "pixel_y", "u", "v_image", "v_obj", "x", "y", "z",],)
            writer.writeheader()
            writer.writerows(representative_points)

    np.save(OUTPUT_DIR / "slic_superpixel_labels.npy", labels)
    np.save(OUTPUT_DIR / "kmeans_cluster_map.npy", cluster_map)
    np.save(OUTPUT_DIR / "texture_xyz_map.npy", xyz_map)
    np.save(OUTPUT_DIR / "valid_texture_mask.npy", valid_mask)
    np.save(OUTPUT_DIR / "superpixel_mean_rgb.npy", mean_rgb)
    np.save(OUTPUT_DIR / "superpixel_mean_lab_adjusted.npy", lab_for_kmeans)
    np.save(OUTPUT_DIR / "cluster_centers_lab_adjusted.npy", centers_lab_adjusted)

    save_superpixel_table(
        OUTPUT_DIR / "superpixel_features.csv",
        mean_rgb=mean_rgb,
        lab_original=lab_original,
        lab_adjusted=lab_for_kmeans,
        mean_xyz=mean_xyz,
        mean_pixel_rc=mean_pixel_rc,
        counts=pixel_counts,
        cluster_for_superpixel=cluster_for_superpixel,
    )
    save_cluster_centers(
        OUTPUT_DIR / "cluster_centers.csv",
        centers_lab_adjusted=centers_lab_adjusted,
        cluster_display_rgb=cluster_display_rgb,
    )

    save_rgba_float_image(
        OUTPUT_DIR / "superpixel_mean_rgb.png",
        rgb=mean_rgb_texture,
        alpha=alpha,
        valid_mask=valid_mask,
    )
    save_rgba_float_image(
        OUTPUT_DIR / "cluster_centroid_rgb_texture.png",
        rgb=cluster_rgb_texture,
        alpha=alpha,
        valid_mask=valid_mask,
    )

    categorical_rgba = make_categorical_cluster_rgba(
        cluster_map=cluster_map,
        valid_mask=valid_mask,
        alpha=alpha,
        number_of_clusters=N_KMEANS_CLUSTERS,
    )
    Image.fromarray(
        np.round(np.clip(categorical_rgba, 0.0, 1.0) * 255.0).astype(np.uint8),
        mode="RGBA",
    ).save(OUTPUT_DIR / "kmeans_cluster_labels.png")

    overlay_labels = labels.copy()
    overlay_labels[~valid_mask] = int(labels[valid_mask].max()) + 1
    boundary_overlay = mark_boundaries(
        rgb,
        overlay_labels,
        color=(1.0, 0.0, 0.0),
        mode="thick",
    )
    boundary_overlay[~valid_mask] = 0.0
    save_rgba_float_image(
        OUTPUT_DIR / "slic_rgb_xyz_boundaries.png",
        rgb=boundary_overlay,
        alpha=alpha,
        valid_mask=valid_mask,
    )

    print("8/8 Displaying article-style pipeline...")
    create_pipeline_figure(
        rgb=rgb,
        alpha=alpha,
        labels=labels,
        valid_mask=valid_mask,
        mean_rgb_texture=mean_rgb_texture,
        categorical_rgba=categorical_rgba,
        cluster_rgb_texture=cluster_rgb_texture,
        representative_points=representative_points,
        output_dir=OUTPUT_DIR,
    )

    print(f"Finished. Outputs saved in: {OUTPUT_DIR}")

    # return vertices, texcoords, triangle_vertex_indices, triangle_uv_indices, xyz_map, uv_mask, hit_count, spatial_step #, ambiguous_mask


if __name__ == "__main__":
    main()